# Concrete Strength Prediction
Predicting concrete compressive strength (MPa) from its ingredients and curing age.

Pipeline: raw Excel file → cleaned Delta table → SQL exploration → machine learning model.

## Step 1: Load and clean the raw data

In [0]:
# Install xlrd (to read legacy .xls) and plotly (interactive charts)
%pip install xlrd plotly

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Import pandas for data loading and cleaning
import pandas as pd

# Load the raw Excel file from the Unity Catalog volume
raw = pd.read_excel("/Volumes/workspace/default/raw_data/Concrete_Data.xls")

# Check size: (rows, columns)
raw.shape

(1030, 9)

In [0]:
# Rename long, messy column names to short, clean ones
raw.columns = ["cement", "slag", "fly_ash", "water", "superplasticizer",
               "coarse_agg", "fine_agg", "age", "strength"]

# Give each mix a permanent ID so row order can always be restored
raw["sample_id"] = range(len(raw))

# Preview to confirm names and IDs line up with the data
raw.head()

,cement,slag,fly_ash,water,superplasticizer,coarse_agg,fine_agg,age,strength,sample_id
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111,0
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366,1
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535,2
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780,3
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075,4


In [0]:
# Check for missing values in every column
print("Missing values per column:")
print(raw.isnull().sum())

# Check for exact duplicate mixes (same recipe, same age, same strength)
dupes = raw.drop(columns=["sample_id"]).duplicated().sum()
print("\nExact duplicate rows:", dupes)

Missing values per column:
cement              0
slag                0
fly_ash             0
water               0
superplasticizer    0
coarse_agg          0
fine_agg            0
age                 0
strength            0
sample_id           0
dtype: int64

Exact duplicate rows: 25


In [0]:
# Convert to a Spark DataFrame and save as a Delta table
df_spark = spark.createDataFrame(raw)
df_spark.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.concrete_clean")


## Step 2: Explore the data with SparkSQL

In [0]:
%sql
-- Validate: total mixes, and mixes that used fly ash
SELECT COUNT(*) AS total_mixes,
       SUM(CASE WHEN fly_ash > 0 THEN 1 ELSE 0 END) AS fly_ash_mixes
FROM workspace.default.concrete_clean

total_mixes,fly_ash_mixes
1030,464


In [0]:
%sql
-- Average strength for each curing age
SELECT age,
       COUNT(*) AS mixes,
       ROUND(AVG(strength), 1) AS avg_strength
FROM workspace.default.concrete_clean
GROUP BY age
ORDER BY age

age,mixes,avg_strength
1,2,9.5
3,134,19.0
7,126,26.1
14,62,28.8
28,425,36.7
56,91,51.9
90,54,40.5
91,22,69.8
100,52,47.7
120,3,39.6


In [0]:
# Correlation only captures straight line relationships
raw.drop(columns=["sample_id"]).corr()["strength"].drop("strength").sort_values(ascending=False).round(3)

cement              0.498
superplasticizer    0.366
age                 0.329
slag                0.135
fly_ash            -0.106
coarse_agg         -0.165
fine_agg           -0.167
water              -0.290
Name: strength, dtype: float64

## Step 3: Train and evaluate models

In [0]:
# Read the clean Delta table back into pandas for scikit-learn
df = spark.table("workspace.default.concrete_clean").toPandas()

# Restore the original row order using the permanent ID
df = df.sort_values("sample_id").reset_index(drop=True)

# Confirm the first rows are back in order
df[["sample_id", "cement", "strength"]].head(3)

,sample_id,cement,strength
0,0,540.0,79.986111
1,1,540.0,61.887366
2,2,332.5,40.269535


In [0]:
from sklearn.model_selection import train_test_split

# X = ingredients and age (the questions), y = strength (the answer key)
# Drop sample_id too, because an ID number is not an ingredient
X = df.drop(columns=["strength", "sample_id"])
y = df["strength"]

# 80% training (homework), 20% testing (exam), same shuffle every time
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(824, 8) (206, 8)


In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Baseline: linear regression assumes straight line relationships
lr = LinearRegression()
lr.fit(X_train, y_train)          # study the homework
lr_pred = lr.predict(X_test)      # take the exam

print("Linear Regression")
print("MAE:", round(mean_absolute_error(y_test, lr_pred), 2))
print("R2:", round(r2_score(y_test, lr_pred), 2))

Linear Regression
MAE: 7.75
R2: 0.63


In [0]:
from sklearn.ensemble import RandomForestRegressor

# Random forest: many decision trees that vote, so it can capture curves
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest")
print("MAE:", round(mean_absolute_error(y_test, rf_pred), 2))
print("R2:", round(r2_score(y_test, rf_pred), 2))

Random Forest
MAE: 3.78
R2: 0.88


In [0]:
# Which ingredients did the random forest rely on most?
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).round(3)

age                 0.333
cement              0.324
water               0.124
slag                0.076
superplasticizer    0.059
fine_agg            0.036
coarse_agg          0.028
fly_ash             0.020
dtype: float64

In [0]:
from sklearn.model_selection import cross_val_score, KFold

# Shuffle the rows, then cut them into 5 equal folds (same shuffle every time)
folds = KFold(n_splits=5, shuffle=True, random_state=42)

# Test each model on 5 different splits instead of just one
lr_scores = cross_val_score(LinearRegression(), X, y, cv=folds, scoring="r2")
rf_scores = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=42),
                            X, y, cv=folds, scoring="r2")

print("Linear Regression R2 per fold:", lr_scores.round(2), "average:", round(lr_scores.mean(), 2))
print("Random Forest R2 per fold:", rf_scores.round(2), "average:", round(rf_scores.mean(), 2))

Linear Regression R2 per fold: [0.63 0.59 0.56 0.68 0.55] average: 0.6
Random Forest R2 per fold: [0.88 0.91 0.91 0.92 0.92] average: 0.91


In [0]:
import plotly.express as px

# Put actual and predicted side by side in a small table
results = pd.DataFrame({"actual": y_test.values, "predicted": rf_pred})

# Interactive scatter: hover over any dot to see its exact values
fig = px.scatter(results, x="actual", y="predicted", opacity=0.6,
                 title="Random Forest: Predicted vs Actual Strength (MPa)",
                 labels={"actual": "Actual strength (MPa)",
                         "predicted": "Predicted strength (MPa)"})

# Add the perfect prediction line
fig.add_shape(type="line", x0=0, y0=0, x1=85, y1=85,
              line=dict(color="red", dash="dash"))

fig.show()

<!doctype html>

In [0]:
# Error = predicted minus actual. Positive = overpredicted, negative = underpredicted
errors = rf_pred - y_test.values

print("Overpredicted:", (errors > 0).sum())
print("Underpredicted:", (errors < 0).sum())
print("Average error:", round(errors.mean(), 2))
print("Biggest overprediction:", round(errors.max(), 1), "MPa")

Overpredicted: 100
Underpredicted: 106
Average error: -0.84
Biggest overprediction: 17.2 MPa


In [0]:
# Save test predictions back to a Delta table: raw file in, predictions out
pred_table = X_test.copy()
pred_table["actual_strength"] = y_test.values
pred_table["predicted_strength"] = rf_pred.round(2)
pred_table["error"] = (rf_pred - y_test.values).round(2)

spark.createDataFrame(pred_table).write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.concrete_predictions")

In [0]:
%sql
-- The 5 biggest overpredictions, the risky ones
SELECT cement, water, age, actual_strength, predicted_strength, error
FROM workspace.default.concrete_predictions
ORDER BY error DESC
LIMIT 5

cement,water,age,actual_strength,predicted_strength,error
313.0,127.0,28,44.51946532,61.72,17.2
312.7,127.3,28,44.523602176000004,61.72,17.2
312.7,178.1,28,25.10382116,41.62,16.51
313.0,178.0,28,25.0969264,41.54,16.44
388.6,157.9,7,34.90127512,45.66,10.76


## Summary of findings
* **Data:** 1,030 concrete mixes, 8 inputs (7 ingredients plus curing age), no missing values, and 25 exact duplicate rows (the same recipe tested more than once)
* **Pipeline:** raw .xls in a Unity Catalog volume → cleaned with pandas → saved as a Delta table → explored with SparkSQL → modeled with scikit learn → predictions saved back as a Delta table
* **Exploration:** strength rises fast in the first month, then levels off (nonlinear). 28 days is the most common test age, the industry standard
* **Correlation:** cement had the strongest straight line relationship with strength (0.50), while age ranked only third (0.33)
* **Baseline:** linear regression, average R2 of 0.60 across 5 fold cross validation
* **Best model:** random forest, average R2 of 0.91, beating the baseline on every fold, with an average error of about 3.8 MPa
* **Top drivers:** age and cement, about two thirds of the model's decisions combined. Age jumped from third in correlation to first in the forest, because its effect on strength curves rather than following a straight line
* **Error check:** predictions were balanced (100 over, 106 under), but the worst misses overpredicted by about 17 MPa, the risky direction
* **Biggest misses:** near identical recipes around 313 kg of cement, which look like repeated tests of the same mix and would be worth reviewing with a materials expert
* **Caveat:** because 25 mixes are exact duplicates, a repeated mix can land in both the training and test sets, which may make scores look slightly better than they would on brand new recipes
* **Lesson learned:** Spark returned rows in a different order, so I added a permanent sample_id to keep results reproducible
* **Next steps:** remove or group duplicate mixes before splitting, add engineered features like the water to cement ratio, track experiments with MLflow, try other models, and apply the same approach to polymer lab data